#  Bonus: Protokół Agent-to-Agent

## Cel: Zrozumiesz architektoniczną różnicę między używaniem narzędzi a współpracą systemów i własnoręcznie wystawisz agenta jako niezależną usługę sieciową gotową do przyjmowania oddelegowanych zadań

### Czego się nauczysz:
1. **Dlaczego "Internet Agentów" to nieunikniona przyszłość** – zrozumiesz moment przejścia od budowania izolowanych, "samotnych" botów do tworzenia sieci współpracujących systemów autonomicznych.
2. **Czym różni się "mięsień" od "partnera"** – nauczysz się fundamentalnego rozróżnienia architektury: kiedy używać MCP (podłączanie narzędzi), a kiedy A2A (delegowanie zadań innemu agentowi).
3. **Kto aktualnie rozdaje karty w standardach** – poznasz strategiczny krajobraz rywalizacji (Google, IBM, Linux Foundation) i dowiesz się, dlaczego nie warto wiązać się z jednym dostawcą ("Vendor Lock-in").
4. **Jak wystawić Agenta na świat** – przejdziesz praktyczny warsztat z pakowania logiki agenta w niezależny serwer HTTP (FastAPI), gotowy do rozmowy z innymi maszynami.
5. **Jak projektować dwuwarstwową strukturę danych** – nauczysz się oddzielać **Internal Schema** (logikę myślenia agenta) od **External Schema** (kontraktu sieciowego), co jest kluczem do stabilności systemu.
6. **Gdzie leżą inżynierskie miny** – zobaczysz surową prawdę o stanie technologii wczesnej fazy (alpha) i zrozumiesz, dlaczego umiejętność "ręcznego sterowania" jest wciąż ważniejsza od ślepego zaufania standardom.
7. **Strategia: kiedy innowacja, a kiedy klasyka** – zdobędziesz narzędzia do oceny na chłodno: kiedy warto inwestować czas w protokoły A2A, a kiedy bezpieczniej i szybciej dowieźć wartość na klasycznym API.


### Wprowadzenie

Budujemy dzisiaj systemy AI, które przypominają genialnych ekspertów zamkniętych w dźwiękoszczelnych pokojach. Masz świetnego agenta: jeden robi A, drugi B, trzeci C itd. Każdy z nich jest doskonały w swojej dziedzinie, ale gdy chcesz, żeby ze sobą współpracowały, zaczyna się "drutowanie". Tworzysz niestandardowe API, piszesz glue-code i modlisz się, żeby format JSON-a się nie zmienił. To klasyczny problem Wieży Babel – wszyscy chcą budować wysoko, ale nikt nie mówi tym samym językiem.

W poprzednich lekcjach omawialiśmy Model Context Protocol (MCP). To obiecujący standard, ale myśl o nim jak o "USB-C dla AI". Dzięki niemu agent może łatwo podpiąć sobie "urządzenie peryferyjne" – kalkulator, bazę danych czy system plików. Ale co w sytuacji, gdy agent nie potrzebuje kalkulatora, tylko innego agenta? Co jeśli chce zlecić skomplikowane zadanie "koledze" z innego działu, który ma własną pamięć, własne narzędzia i własną autonomię? Tutaj wtyczka USB nie wystarczy. Tutaj potrzebujemy protokołu sieciowego.

Wchodzimy na terytorium "internetu Agentów". Google promuje standard [A2A (Agent-to-Agent)](https://developers.googleblog.com/en/a2a-a-new-era-of-agent-interoperability/), IBM stawia na [ACP](https://research.ibm.com/blog/multiagent-bee-ai), a społeczność open-source szuka własnych dróg. Brutalna prawda jest taka: to technologia we wczesnej fazie alpha. Nie jest to jeszcze gotowe na produkcję mission-critical. Jednak jako inżynierowie Software 3.0 musimy trzymać rękę na pulsie. Zobaczmy więc, jak wygląda przyszłość komunikacji między systemami, zanim stanie się to mainstreamem.

## Krajobraz protokołów: `MCP`,  `A2A/ACP` oraz `AG-UI`

Rok 2025 to moment zwrotny. Przestajemy skupiać się wyłącznie na tym, "jak napisać prompt", a zaczynamy rozwiązywać problem: "jak to wszystko połączyć, żeby nie wybuchło". Do tej pory budowaliśmy silosy. Twój agent w Pythonie świetnie działał w terminalu, ale wystawienie go na świat, podłączenie do innych agentów czy zbudowanie reaktywnego UI oznaczało pisanie setek linii "glue-code" (kodu sklejającego).

Dobra wiadomość jest taka, że branża (Google, IBM, Anthropic) zauważyła ten bałagan i – co rzadkie – dąży do standaryzacji, a nie fragmentacji. Wyłaniają się trzy fundamentalne filary, które adresują zupełnie inne warstwy problemu.

### Mapa drogowa standardów

Spójrzmy na to "na chłodno". Nie musisz wybierać jednego zwycięzcy, bo te protokoły się uzupełniają, a nie wykluczają.

| Protokół | Główny gracz / Opiekun | Warstwa | Rola Inżynierska |
| :--- | :--- | :--- | :--- |
| `MCP` (*Model Context Protocol*) | **Anthropic** | **Dane i narzędzia** 🔌 | "USB dla Agenta". Standaryzacja dostępu do plików, baz danych i prostych API. |
| `A2A / ACP` (*Agent-to-Agent*) | **Google / IBM / Linux Foundation** | **Backend (orkiestracja)** 🌐 | "HTTP dla Agenta". Protokół pozwalający agentom na negocjacje, delegowanie zadań i discovery. |
| `AG-UI` (*Agent-User Interaction*) | **CopilotKit** | **Frontend (użytkownik)** 🖥️ | "HDMI dla Agenta". Zarządzanie stanem, strumieniowaniem i generatywnym interfejsem dla człowieka. |



### Co kryje się pod maską?

#### 1. `MCP` (Tył zaplecza) – Warstwa wykonawcza
To standard dla "mięśni" agenta. Dzięki MCP nie musisz pisać osobnej integracji dla Google Drive, osobnej dla Slacka i osobnej dla Postgresa. Piszesz jeden konektor MCP i Twój agent (niezależnie czy to Claude, czy Llama) widzi te zasoby jako narzędzia. To jest warstwa **stateless** – proste żądanie i odpowiedź.

#### 2. `A2A` oraz `ACP` (Sala konferencyjna) – Warstwa zarządcza
Tutaj robi się ciekawie. Do niedawna mieliśmy dwóch rywali:
*   **A2A (Google):** Skupiony na "Service Discovery" i delegowaniu zadań.
*   **ACP (IBM / BeeAI):** Oparty na REST, bardzo pragmatyczny protokół (Agent Communication Protocol), który traktuje agentów jak mikroserwisy. Pozwala na wymianę dowolnych typów danych (MimeTypes) i obsługę procesów asynchronicznych (long-running tasks).

### ☝️Zwróć uwagę 
`ACP` stał się częścią inicjatywy `A2A` pod parasolem Linux Foundation.
Co to oznacza dla Ciebie? Że nie musisz martwić się "Vendor Lock-in". IBM i Google tworzą wspólny standard, gdzie agent napisany w frameworku `BeeAI` (IBM) będzie mógł bezboleśnie "zatrudnić" agenta napisanego w `Vertex AI` (Google). To jest prawdziwy "Internet Agentów" – oparty na REST, otwarty i niezależny od tego, czy pod spodem masz Pythona, czy Node.js.



## `MCP` vs `A2A`
```

╔═══════════════════════════════════════════════════════════════════════════════════╗
║                                                                                   ║
║   🔧 MCP (Model Context Protocol)                                                 ║
║   ─────────────────────────────────                                               ║
║   "Agent talks to TOOLS and DATA"                                                 ║
║                                                                                   ║
║                                                                                   ║
║                            ┌─────────────────┐                                    ║
║                            │   📁 Files      │                                    ║
║                            │   (Google Drive)│                                    ║
║                            └────────▲────────┘                                    ║
║                                     │                                             ║
║                                     │ MCP                                         ║
║                                     │                                             ║
║      ┌─────────────────┐    ┌───────┴───────┐    ┌─────────────────┐              ║
║      │   🗄️ Database   │◄───┤               ├───►│   🔌 External   │              ║
║      │                 │MCP │   🤖 AGENT    │ MCP│      APIs       │              ║
║      └─────────────────┘    │               │    └─────────────────┘              ║
║                             └───────┬───────┘                                     ║
║                                     │                                             ║
║                                     │ MCP                                         ║
║                                     │                                             ║
║                            ┌────────▼────────┐                                    ║
║                            │   💻 Terminal   │                                    ║
║                            │   (commands)    │                                    ║
║                            └─────────────────┘                                    ║
║                                                                                   ║
║   ✅ One agent, many tools                                                        ║
║   ✅ Standardized access to resources                                             ║
║   ✅ "USB-C for AI"                                                               ║
║                                                                                   ║
╠═══════════════════════════════════════════════════════════════════════════════════╣
║                                                                                   ║
║   🤝 A2A (Agent-to-Agent Protocol)                                                ║
║   ─────────────────────────────────                                               ║
║   "Agent talks to OTHER AGENTS"                                                   ║
║                                                                                   ║
║                                                                                   ║
║   ┌──────────────┐         ┌──────────────┐         ┌──────────────┐              ║
║   │  🤖 TRAVEL   │   A2A   │  🤖 PERSONAL │   A2A   │  🤖 BANKING  │              ║
║   │    Agent     │◄───────►│    Agent     │◄───────►│    Agent     │              ║
║   │              │         │   (Yours)    │         │              │              ║
║   │ "Searching   │         │              │         │ "Checking    │              ║
║   │  for hotel"  │         │ "Plan my     │         │  budget"     │              ║
║   └──────────────┘         │  vacation"   │         └──────────────┘              ║
║                            └──────┬───────┘                                       ║
║                                   │                                               ║
║                                   │ A2A                                           ║
║                                   │                                               ║
║                            ┌──────▼───────┐                                       ║
║                            │  🤖 FLIGHT   │                                       ║
║                            │    Agent     │                                       ║
║                            │              │                                       ║
║                            │ "Booking     │                                       ║
║                            │  tickets"    │                                       ║
║                            └──────────────┘                                       ║
║                                                                                   ║
║   ✅ Multiple agents collaborate                                                  ║
║   ✅ Each agent has its own specialization                                        ║
║   ✅ Agents can be from different vendors                                         ║
║                                                                                   ║
╠═══════════════════════════════════════════════════════════════════════════════════╣
║                                                                                   ║
║   🔄 TOGETHER - Full Ecosystem                                                    ║
║   ────────────────────────────                                                    ║
║                                                                                   ║
║        A2A LAYER (agent-to-agent communication)                                   ║
║   ┌─────────────────────────────────────────────────────────────┐                 ║
║   │                                                             │                 ║
║   │    🤖 Agent A ◄────► 🤖 Agent B ◄────► 🤖 Agent C           │                 ║
║   │         │                  │                  │             │                 ║
║   └─────────┼──────────────────┼──────────────────┼─────────────┘                 ║
║             │                  │                  │                               ║
║        MCP LAYER (resource access)                                                ║
║   ┌─────────┼──────────────────┼──────────────────┼─────────────┐                 ║
║   │         │                  │                  │             │                 ║
║   │         ▼                  ▼                  ▼             │                 ║
║   │       📁🗄️               🔌💳               📅📧           │                 ║
║   │    Files, DB         Payment APIs      Calendar, Email      │                 ║
║   │                                                             │                 ║
║   └─────────────────────────────────────────────────────────────┘                 ║
║                                                                                   ║
╚═══════════════════════════════════════════════════════════════════════════════════╝

```



### Inżynieria wertykalna vs horyzontalna

Spójrz na powyższy schemat. To najlepszy sposób, by zbudować sobie mentalny model tego, jak te technologie ze sobą współgrają. Nie musisz wybierać "albo-albo".

1. `MCP` (góra diagramu) – To są Twoje ręce. Działa wertykalnie (pionowo). To relacja typu **Agent-Narzędzie**.
    - Agent jest tutaj "mózgiem", a baza danych czy plik to pasywny zasób.
    - Komunikacja jest deterministyczna: "Daj mi plik", "Zapisz rekord".
    - **Kluczowa cecha:** Narzędzia nie mają własnej agendy ani autonomii. Czekają na polecenie.

2. `A2A` (środek diagramu) – To jest Twój zespół. Działa horyzontalnie (poziomo). To relacja typu **Agent-Agent**.
    - Tutaj rozmawiają ze sobą "równorzędni partnerzy". Twój "Personal Agent" nie musi wiedzieć, jak działa system rezerwacji lotów. On po prostu deleguje zadanie do specjalisty ("Flight Agent").
    - **Kluczowa cecha:** Każdy agent w tej sieci ma własną autonomię, własny stan i własne cele. To nie jest wywołanie funkcji, to negocjacja.

3. Pełny ekosystem (dół diagramu) – warstwowa architektura. To jest docelowa architektura systemów Software 3.0.
    - *Warstwa zarządzania (`A2A`):* Agenci komunikują się między sobą, przekazując zadania i wyniki (np. Agent Sprzedaży prosi Agenta Magazynu o sprawdzenie stanu).
    - *Warstwa wykonawcza (`MCP`):* Gdy Agent Magazynu dostanie zadanie, używa `MCP`, by fizycznie odpytać bazę SQL.

`MCP` służy do wykonywania pracy (interakcja ze światem cyfrowym), a `A2A` służy do organizowania pracy (interakcja między systemami decyzyjnymi).

#### 3. AG-UI (scena frontowa) – warstwa prezentacji
Tradycyjne REST API nie radzi sobie z tym, że agent chce nagle, w połowie zdania, wyświetlić wykres (Generative UI) albo zapytać użytkownika o zgodę.

**AG-UI** to "trzecia noga", która łączy mózg agenta z oczami użytkownika. To protokół oparty na zdarzeniach (events), który mówi frontendowi: *"Hej, teraz strumieniuję tekst, a teraz renderuję komponent Reacta, a teraz czekam na kliknięcie"*. Jeśli budujesz aplikację dla ludzi, a nie dla botów, MCP i A2A Ci nie wystarczą – potrzebujesz `AG-UI`.


```
╔═══════════════════════════════════════════════════════════════════════════════════╗
║                                                                                   ║
║   🖥️ AG-UI (Agent-User Interface Protocol)                                        ║
║   ─────────────────────────────────────────                                       ║
║   "Agent talks to the USER through the interface"                                 ║
║                                                                                   ║
║                                                                                   ║
║   ┌─────────────────────────────────────────────────────────────────────────┐     ║
║   │                         👤 USER                                         │     ║
║   │                                                                         │     ║
║   │   "Book me a flight to Tokyo and show me hotel options"                 │     ║
║   └─────────────────────────────────┬───────────────────────────────────────┘     ║
║                                     │                                             ║
║                                     │ clicks, types, speaks                       ║
║                                     ▼                                             ║
║   ┌─────────────────────────────────────────────────────────────────────────┐     ║
║   │                                                                         │     ║
║   │                        🖥️ USER INTERFACE                                │     ║
║   │                                                                         │     ║
║   │  ┌─────────────┐  ┌─────────────┐  ┌─────────────┐  ┌─────────────┐     │     ║
║   │  │ 💬 Chat     │  │ 📊 Charts   │  │ 📝 Forms    │  │ ⏳ Progress │     │     ║
║   │  │   bubbles   │  │   & tables  │  │   & inputs  │  │   indicators│     │     ║
║   │  └─────────────┘  └─────────────┘  └─────────────┘  └─────────────┘     │     ║
║   │                                                                         │     ║
║   │  ┌─────────────┐  ┌─────────────┐  ┌─────────────┐  ┌─────────────┐     │     ║
║   │  │ ✅ Confirm  │  │ 🔔 Notifi-  │  │ 🖼️ Rich     │  │ 🎛️ Inter-   │     │     ║
║   │  │   buttons   │  │   cations   │  │   media     │  │   active    │     │     ║
║   │  └─────────────┘  └─────────────┘  └─────────────┘  └─────────────┘     │     ║
║   │                                                                         │     ║
║   └─────────────────────────────────┬───────────────────────────────────────┘     ║
║                                     │                                             ║
║                                     │ AG-UI Protocol                              ║
║                                     │ (streaming, events, state)                  ║
║                                     ▼                                             ║
║   ┌─────────────────────────────────────────────────────────────────────────┐     ║
║   │                                                                         │     ║
║   │                          🤖 AI AGENT                                    │     ║
║   │                                                                         │     ║
║   │    Sends structured events:                                             │     ║
║   │                                                                         │     ║
║   │    📤 TEXT_STREAM      → "Searching for flights..."                     │     ║
║   │    📤 TOOL_START       → "Calling flight API"                           │     ║
║   │    📤 TOOL_RESULT      → {flights: [...]}                               │     ║
║   │    📤 STATE_UPDATE     → {step: 2, total: 5}                            │     ║
║   │    📤 USER_CONFIRM     → "Book this flight for $450?"                   │     ║
║   │    📤 RENDER_COMPONENT → <FlightCard data={...} />                      │     ║
║   │                                                                         │     ║
║   └─────────────────────────────────────────────────────────────────────────┘     ║
║                                                                                   ║
║   ✅ Real-time streaming of responses                                             ║
║   ✅ Rich UI components (not just text)                                           ║
║   ✅ Human-in-the-loop confirmations                                              ║
║   ✅ Progress visibility ("Agent is thinking...")                                 ║
║                                                                                   ║
╚═══════════════════════════════════════════════════════════════════════════════════╝
```


### `AG-UI` (Interfejs Agent-Użytkownik)

Ten diagram obrazuje fundamentalną zmianę w projektowaniu frontendów. W klasycznym modelu (REST API) serwer zwracał dane (JSON), a frontend decydował, jak je wyświetlić. W modelu **Software 3.0** to **Agent steruje interfejsem**.

Patrząc od góry do dołu:

1. Użytkownik 
    - Wysyła intencję ("Zarezerwuj lot...")
    - To nie jest tylko tekst – to mogą być kliknięcia, wypełnione formularze czy interakcje głosowe

2. Interfejs (User Interface)
    - Zwróć uwagę na bloki w środku: "Chat bubbles", "Charts", "Confirm buttons".
    - Interfejs jest dynamiczny. Nie jest sztywną makietą. Składa się z komponentów (klocków), które mogą się pojawiać i znikać w zależności od tego, co "wymyśli" agent.

3. Protokół
    - Strzałka w dół symbolizuje *streaming*. To nie jest jedno zapytanie i jedna odpowiedź. To ciągły strumień zdarzeń.
    - To tutaj dzieje się magia `AG-UI`. Protokół przesyła stan, aktualizacje i żądania renderowania w czasie rzeczywistym.

4.  AI Agent)
    - To jest najważniejsza część diagramu. Zobacz, co wysyła agent. To nie są tylko ciągi znaków (String). To **ustrukturyzowane zdarzenia (Structured Events)**:
        *   `📤 TOOL_START`: Frontend wie, że ma wyświetlić "spinner" ładowania (bo agent dzwoni do API).
        *   `📤 USER_CONFIRM`: Agent wymusza na interfejsie wyświetlenie przycisku "Zatwierdź / Odrzuć" (Human-in-the-Loop).
        *   `📤 RENDER_COMPONENT`: Agent mówi wprost: "Wyświetl mi tutaj kartę lotu (FlightCard) z tymi danymi". To jest tzw. **Generative UI**.

AG-UI zamienia "Chatbota" w pełnoprawną "Aplikację sterowaną rozmową".


### Trzy protokoły obok siebie

```
┌─────────────────────────────────────────────────────────────────────────────────┐
│                                                                                 │
│   PROTOCOL        DIRECTION              PURPOSE              ANALOGY           │
│   ─────────────────────────────────────────────────────────────────────────     │
│                                                                                 │
│                        ▲                                                        │
│   🖥️ AG-UI        Agent → User       Present & interact     📺 TV Screen        │
│                        │               with humans           (what you see)     │
│                        │                                                        │
│   ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─       │
│                        │                                                        │
│                   🤖 AGENT                                                      │
│                        │                                                        │
│   ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─       │
│                        │                                                        │
│                   ┌────┴────┐                                                   │
│                   │         │                                                   │
│                   ▼         ▼                                                   │
│                                                                                 │
│   🔧 MCP      Agent → Tools      Access external          🔌 USB-C              │
│                                   resources               (plug & play)         │
│                                                                                 │
│   🤝 A2A      Agent ↔ Agent      Collaborate with         📞 Phone call         │
│                                   other agents            (conversation)        │
│                                                                                 │
└─────────────────────────────────────────────────────────────────────────────────┘

```


### Trzy protokoły (widok z lotu ptaka)

Ten diagram porządkuje architekturę przestrzenną. Pokazuje, w którą stronę "patrzy" agent w zależności od zadania.

Centralnym punktem jest **AGENT (Mózg systemu)**.

1.  Kierunek "W GÓRĘ" (Front-of-House) 🔼
    - *Protokół:* `🖥️ AG-UI`
    - *Cel:* Prezentacja i interakcja
    - *Odbiorca:* Człowiek
    - *Analogia:* *Telewizor*. To jest to, co widzi użytkownik końcowy. To fasada, która musi być ładna, responsywna i zrozumiała.

2. Kierunek "W DÓŁ - LEWO" (Back-of-House - zasoby) 🔽
    - *Protokół:* `🔧 MCP`
    - *Cel:* Wykonanie pracy (*Execution*)
    - *Odbiorca:* Narzędzia (Bazy danych, API, Pliki).
    - *Analogia:* **Wtyczka USB*. Podłączasz dysk twardy (Tools) do komputera (Agent). Dysk nie myśli, dysk wykonuje polecenia zapisu/odczytu. To relacja: Pan (Agent) – Sługa (Narzędzie).

3.  Kierunek "W DÓŁ - PRAWO" (Back-of-House - Współpraca) 🔽**
    - *Protokół:** `🤝 A2A` (oraz ACP)
    - *Cel:* Delegacja i współpraca
    - *Odbiorca:* Inne Agenty AI
    - *Analogia:* **Rozmowa telefoniczna**. Twój agent dzwoni do innego agenta (np. specjalisty od bankowości). To relacja partnerska. Tamten agent ma swój własny "mózg", swoje narzędzia i może odmówić wykonania zadania lub negocjować parametry.


Budując system Software 3.0, stoisz w samym środku tego diagramu. To jest Twój kokpit:
- Używasz **AG-UI**, żeby "dogadać się" z człowiekiem (User Experience).
- Używasz **MCP**, żeby "mieć ręce" do pracy na danych (Execution).
- Używasz **A2A**, żeby "mieć zespół" do pomocy przy złożonych problemach (Delegation).

Wyobraź sobie taki scenariusz w systemie e-commerce:
1.  Użytkownik pisze na czacie: "Znajdź mi te buty, ale taniej" (**AG-UI** obsługuje strumieniowanie i interfejs).
2.  Twój główny agent sprawdza stany magazynowe w Twoim SAP (**MCP** daje mu bezpieczny dostęp do bazy).
3.  Okazuje się, że towaru brak, więc agent zleca zewnętrznemu agentowi logistycznemu znalezienie zamiennika u partnerów. Agent logistyczny to obca usługa, w innej firmie (**A2A** obsługuje handshake, autoryzację i przekazanie zadania).

Bez tych standardów musiałbyś "odrutować" każde z tych połączeń ręcznie, tworząc dziesiątki niestandardowych API. Z protokołami – teoretycznie – podmieniasz klocki.

### ⚠️ Brutalna prawda: wizja vs rzeczywistość

Tutaj musimy się zatrzymać i włączyć inżynierski krytycyzm. Skoro te protokoły są tak wspaniałe, to dlaczego w tym module uczyliśmy się ręcznej kontroli nad wywoływaniem narzędzi i budowania własnych struktur?

**Bo jesteśmy w fazie turbulencji.**

Musisz rozróżnić dwie rzeczy:
1.  **Gotowość mentalna (*Mental Models*):** Koncepty podziału na `AG-UI`, `MCP` i `A2A` są **dojrzałe**. Już teraz powinieneś projektować systemy w głowie, używając tych kategorii. Nie możesz ich ignorować, bo tak będzie wyglądać architektura systemów AI przez najbliższą dekadę.
2.  **Gotowość wdrożeniowa (*Implementation*):** Tutaj panuje "Dziki Zachód". Standardy "walczą" ze sobą (Google vs IBM), biblioteki są w wersji `0.1.0`, a dokumentacja często nie nadąża za kodem.

### Dlaczego uczymy "ręcznej kontroli"?

Właśnie dlatego w tym module kładliśmy tak duży nacisk na **stabilizację** i przejmowanie kontroli nad procesem wywoływania narzędzi.

Kiedy standardy (jak `A2A`) są jeszcze dziurawe, Twoim obowiązkiem jako inżyniera jest zasypywanie tych luk. Używasz gotowych konceptów (wiesz, że chcesz połączyć agentów), ale na poziomie implementacji musisz wciąż "trzymać kierownicę", dopóki autopilot (standardy) nie stanie się w pełni niezawodny.

Ucz się tych protokołów. Eksperymentuj z nimi. Ale na produkcję zabieraj ze sobą własne pasy bezpieczeństwa, które zbudowaliśmy w poprzednich lekcjach.

### Krok 1: Wczytujemy biblioteki

In [ ]:
import uvicorn
import asyncio
import threading
import time
import httpx
import uuid
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any

from enum import Enum
import helper as h

### Krok 2: Definicja struktur

Tutaj dzieje się najważniejsza inżynierska praca. Musimy rozróżnić dwa typy kontraktów:
1.  *Internal Schema:* Co model AI zwraca wewnątrz funkcji (logika biznesowa).
2.  *External Schema* (A2A):** Co nasz serwer wystawia na świat (protokół komunikacyjny).

Stworzymy agenta *"Ticket Triage"*. Jego zadaniem jest kategoryzacja zgłoszeń od klientów. Inne agenty (np. Agent Sprzedaży) będą mu wysyłać e-maile, a on ma zwrócić JSON z priorytetem i działem.

In [ ]:
# A. Internal Schema (Logika Agenta)
# -----------------------------------

class TicketPriority(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"

class TicketCategory(str, Enum):
    BILLING = "billing"
    TECHNICAL = "technical"
    SALES = "sales"
    OTHER = "other"

class TriageResult(BaseModel):
    category: TicketCategory = Field(..., description="Dział, do którego należy zgłoszenie")
    priority: TicketPriority = Field(..., description="Ocena pilności zgłoszenia")
    reasoning: str = Field(..., description="Krótkie uzasadnienie decyzji (max 1 zdanie)")


In [ ]:
# B. External Schema (Protokół A2A)
# -----------------------------------

class AgentCapability(BaseModel):
    """Wizytówka agenta - Endpoint /info"""
    name: str
    description: str
    version: str = "1.0.0"

class TaskRequest(BaseModel):
    """Koperta wejściowa - Endpoint /tasks"""
    task_id: str
    input_text: str = Field(..., description="Treść zgłoszenia do analizy")

class TaskResponse(BaseModel):
    """Koperta wyjściowa - Co zwracamy po sieci"""
    task_id: str
    status: str = "completed"
    result: TriageResult # <-- Zagnieżdżamy tutaj wynik z LLM!

print("✅ Zdefiniowano kontrakty (Internal & External Pydantic Models)")

### Krok 3: Serwer Agenta (FastAPI)

Teraz spinamy to w całość. Tworzymy aplikację FastAPI. Zwróć uwagę na funkcję `execute_task`. To tutaj następuje "tłumaczenie":
`Request HTTP` -> `Prompt` -> `Structured LLM Output` -> `Response HTTP`.


In [ ]:
app = FastAPI(title="Triage Agent Server")

# Logika biznesowa Agenta
def run_triage_agent(content: str) -> TriageResult:
    messages = [
        {
            "role": "system", 
            "content": "Jesteś ekspertem obsługi klienta (Triage Agent). Analizuj zgłoszenia i nadawaj im kategorie."
        },
        {
            "role": "user", 
            "content": f"Treść zgłoszenia: {content}"
        }
    ]
    
    # Używamy naszej funkcji helpera z Instructorem
    return h.call_api(messages, response_model=TriageResult)

# Endpointy A2A
# -------------

@app.get("/a2a/info", response_model=AgentCapability)
async def get_agent_info():
    return AgentCapability(
        name="TriageBot-v1",
        description="Analizuje tekst zgłoszenia i zwraca kategorię oraz priorytet."
    )

@app.post("/a2a/tasks", response_model=TaskResponse)
async def execute_task(request: TaskRequest):
    print(f"📥 [SERVER] Otrzymano zadanie {request.task_id[:8]}...")
    
    try:
        # 1. Wywołanie logiki (LLM)
        ai_result = run_triage_agent(request.input_text)
        
        # 2. Zapakowanie w kopertę protokołu
        return TaskResponse(
            task_id=request.task_id,
            result=ai_result.model_dump() 
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print("✅ Zdefiniowano serwer Agenta")

### Krok 4: Uruchomienie w tle

Standardowa procedura dla Jupyter Notebooka. Odpalamy serwer na osobnym wątku, żeby nie zablokować sobie możliwości pisania kodu klienta.

In [ ]:
# Konfiguracja serwera
config = uvicorn.Config(app, host="127.0.0.1", port=8999, log_level="error")
server = uvicorn.Server(config)

def start_server():
    server.run()

# Uruchomienie wątku
server_thread = threading.Thread(target=start_server)
server_thread.daemon = True
server_thread.start()

# Czekamy chwilę na start
time.sleep(2)
print("🚀 Serwer A2A (Triage Agent) nasłuchuje na porcie 8999")

## Klient: rozmowa maszyn 🤖💬🤖

Teraz wcielasz się w rolę innego systemu (np. Agenta CRM). Twoim zadaniem jest wysłać surowe zgłoszenie klienta do naszego nowego Agenta Triage i odebrać ustrukturyzowaną decyzję.

Zobacz, jak czysta jest ta komunikacja. Klient nie musi wiedzieć, że tam pod spodem jest GPT-4o. On po prostu wysyła JSON i dostaje JSON.


```

CLIENT                    SERVER (:8999)                  LLM (OpenAI)
       │                            │                              │
       │  POST /a2a/tasks           │                              │
       │  {task_id, input_text}     │                              │
       │ ──────────────────────────►│                              │
       │                            │                              │
       │                            │  messages + TriageResult     │
       │                            │ ────────────────────────────►│
       │                            │                              │
       │                            │      structured JSON         │
       │                            │ ◄────────────────────────────│
       │                            │                              │
       │  {task_id, status, result} │                              │
       │ ◄──────────────────────────│                              │
       │                            │                              │
       │  ✅ category: technical    │                              │
       │  ✅ priority: critical     │                              │
       │  ✅ reasoning: "..."       │                              │
       ▼                            ▼                              ▼
       
```


In [ ]:
async def simulate_agent_conversation():
    async with httpx.AsyncClient() as client:
        
        # Scenariusz: Wpadł wściekły mail od klienta
        customer_email = "Mój serwer produkcyjny padł 3 godziny temu! Tracimy pieniądze! Żądam kontaktu natychmiast!"
        
        task_payload = {
            "task_id": str(uuid.uuid4()),
            "input_text": customer_email
        }
        
        print(f"📨 [CLIENT] Wysyłam zgłoszenie: '{customer_email[:50]}...'")
        
        # Strzał do serwera
        response = await client.post("http://127.0.0.1:8999/a2a/tasks", json=task_payload)
        
        if response.status_code == 200:
            data = response.json()
            # Parsujemy wynik (w prawdziwym kodzie uzylibysmy TaskResponse(**data))
            result = data['result']
            
            print("\n✅ [CLIENT] Odebrano decyzję:")
            print(f"   📂 Kategoria: {result['category']}")
            print(f"   🔥 Priorytet: {result['priority']}")
            print(f"   🧠 Uzasadnienie: {result['reasoning']}")
        else:
            print(f"❌ Błąd: {response.text}")

# Uruchomienie symulacji
await simulate_agent_conversation()

### 🧠 Co tu się właśnie wydarzyło?

To jest moment "Aha!". Zobacz różnicę:

1.  Bez `A2A`: Twój kod musiałby bezpośrednio importować bibliotekę OpenAI i mieć w sobie prompt. Każda zmiana w logice kategoryzacji wymagałaby wdrażania zmian w Twoim głównym kodzie.
2.  Z `A2A`: Twój kod po prostu wysyła request HTTP. Logika kategoryzacji (prompt, model, parsowanie) jest zamknięta w osobnym "pudełku" (kontenerze).

```
╔═══════════════════════════════════════════════════════════════════════════════════════╗
║                                                                                       ║
║   🎯 TRIAGE AGENT - A2A Protocol Flow                                                 ║
║                                                                                       ║
╠═══════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                       ║
║   STEP 1: CLIENT SENDS REQUEST                                                        ║
║   ────────────────────────────────                                                    ║
║                                                                                       ║
║   ┌─────────────────────────────────────────────────────────────────────────────┐     ║
║   │  👤 ANGRY CUSTOMER EMAIL                                                    │     ║
║   │                                                                             │     ║
║   │  "My production server crashed 3 hours ago! We're losing money!             │     ║
║   │   I demand contact immediately!"                                            │     ║
║   └──────────────────────────────────┬──────────────────────────────────────────┘     ║
║                                      │                                                ║
║                                      ▼                                                ║
║   ┌─────────────────────────────────────────────────────────────────────────────┐     ║
║   │  🖥️ CLIENT APPLICATION                                                      │     ║
║   │                                                                             │     ║
║   │  Wraps email in A2A ENVELOPE (TaskRequest):                                 │     ║
║   │  ┌────────────────────────────────────────────┐                             │     ║
║   │  │  {                                         │                             │     ║
║   │  │    "task_id": "1de4232e-...",              │                             │     ║
║   │  │    "input_text": "My production server..." │                             │     ║
║   │  │  }                                         │                             │     ║
║   │  └────────────────────────────────────────────┘                             │     ║
║   └──────────────────────────────────┬──────────────────────────────────────────┘     ║
║                                      │                                                ║
║                                      │  POST /a2a/tasks                               ║
║                                      │  (HTTP Request)                                ║
║                                      ▼                                                ║
╠═══════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                       ║
║   STEP 2: SERVER RECEIVES & UNPACKS                                                   ║
║   ─────────────────────────────────────                                               ║
║                                                                                       ║
║   ┌─────────────────────────────────────────────────────────────────────────────┐     ║
║   │  🤖 TRIAGE AGENT SERVER (FastAPI @ port 8999)                               │     ║
║   │                                                                             │     ║
║   │  ┌─────────────────────────────────────────────────────────────────────┐    │     ║
║   │  │  📥 ENDPOINT: /a2a/tasks                                            │    │     ║
║   │  │                                                                     │    │     ║
║   │  │  1. Receives TaskRequest (A2A envelope)                             │    │     ║
║   │  │  2. Extracts input_text                                             │    │     ║
║   │  │  3. Calls run_triage_agent()                                        │    │     ║
║   │  └──────────────────────────────────┬──────────────────────────────────┘    │     ║
║   │                                     │                                       │     ║
║   └─────────────────────────────────────┼───────────────────────────────────────┘     ║
║                                         │                                             ║
║                                         ▼                                             ║
╠═══════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                       ║
║   STEP 3: LLM PROCESSING (Internal Logic)                                             ║
║   ───────────────────────────────────────                                             ║
║                                                                                       ║
║   ┌─────────────────────────────────────────────────────────────────────────────┐     ║
║   │  🧠 run_triage_agent() - BUSINESS LOGIC                                     │     ║
║   │                                                                             │     ║
║   │  ┌─────────────────────┐         ┌─────────────────────────────────────┐    │     ║
║   │  │  📝 SYSTEM PROMPT   │         │  🎯 INTERNAL SCHEMA                 │    │     ║
║   │  │                     │         │     (TriageResult)                  │    │     ║
║   │  │  "You are a        │          │                                     │    │     ║
║   │  │   customer service │          │  class TriageResult:                │    │     ║
║   │  │   expert (Triage   │          │    category: TicketCategory         │    │     ║
║   │  │   Agent)..."       │          │      • billing                      │    │     ║
║   │  └──────────┬──────────┘         │      • technical  ◄── selected      │    │     ║
║   │             │                    │      • sales                        │    │     ║
║   │             │                    │      • other                        │    │     ║
║   │             ▼                    │                                     │    │     ║
║   │  ┌──────────────────────┐        │    priority: TicketPriority         │    │     ║
║   │  │                      │        │      • low                          │    │     ║
║   │  │   🤖 LLM (OpenAI)    │        │      • medium                       │    │     ║
║   │  │                      │        │      • high                         │    │     ║
║   │  │   + Instructor       │───────►│      • critical  ◄── selected       │    │     ║
║   │  │   (Structured        │        │                                     │    │     ║
║   │  │    Output)           │        │    reasoning: str                   │    │     ║
║   │  │                      │        │      "Server outage requires        │    │     ║
║   │  └──────────────────────┘        │       immediate intervention"       │    │     ║
║   │                                  └─────────────────────────────────────┘    │     ║
║   └─────────────────────────────────────┬───────────────────────────────────────┘     ║
║                                         │                                             ║
║                                         │  Returns TriageResult                       ║
║                                         ▼                                             ║
╠═══════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                       ║
║   STEP 4: PACK RESULT INTO A2A ENVELOPE                                               ║
║   ─────────────────────────────────────────                                           ║
║                                                                                       ║
║   ┌─────────────────────────────────────────────────────────────────────────────┐     ║
║   │  📤 SERVER WRAPS RESPONSE                                                   │     ║
║   │                                                                             │     ║
║   │  EXTERNAL SCHEMA (TaskResponse) - A2A Protocol Envelope:                    │     ║
║   │  ┌────────────────────────────────────────────────────────────────────┐     │     ║
║   │  │  {                                                                 │     │     ║
║   │  │    "task_id": "1de4232e-...",                                      │     │     ║
║   │  │    "status": "completed",                                          │     │     ║
║   │  │    "result": {                          ◄── TriageResult nested    │     │     ║
║   │  │      "category": "technical",                                      │     │     ║
║   │  │      "priority": "critical",                                       │     │     ║
║   │  │      "reasoning": "Server outage requires immediate intervention"  │     │     ║
║   │  │    }                                                               │     │     ║
║   │  │  }                                                                 │     │     ║
║   │  └────────────────────────────────────────────────────────────────────┘     │     ║
║   └──────────────────────────────────┬──────────────────────────────────────────┘     ║
║                                      │                                                ║
║                                      │  HTTP 200 Response                             ║
║                                      ▼                                                ║
╠═══════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                       ║
║   STEP 5: CLIENT RECEIVES DECISION                                                    ║
║   ────────────────────────────────────                                                ║
║                                                                                       ║
║   ┌─────────────────────────────────────────────────────────────────────────────┐     ║
║   │  🖥️ CLIENT PARSES RESPONSE                                                  │     ║
║   │                                                                             │     ║
║   │  ┌──────────────────────────────────────────────────────────────────┐       │     ║
║   │  │                                                                  │       │     ║
║   │  │   ✅ TRIAGE DECISION RECEIVED:                                   │       │     ║
║   │  │                                                                  │       │     ║
║   │  │   📂 Category:  TECHNICAL    → Route to Tech Support Team        │       │     ║
║   │  │   🔥 Priority:  CRITICAL     → Immediate attention required      │       │     ║
║   │  │   🧠 Reasoning: "Server outage requires immediate intervention"  │       │     ║
║   │  │                                                                  │       │     ║
║   │  └──────────────────────────────────────────────────────────────────┘       │     ║
║   │                                                                             │     ║
║   │  → Now client can auto-route ticket to correct team with right priority     │     ║
║   │                                                                             │     ║
║   └─────────────────────────────────────────────────────────────────────────────┘     ║
║                                                                                       ║
╚═══════════════════════════════════════════════════════════════════════════════════════╝

```

Możesz teraz wymienić model w Agencie Triage na lepszy, zmienić prompt, a nawet przepisać go na inny język programowania – dopóki zgadza się kontrakt JSON (Schema), reszta systemu nawet tego nie zauważy. To jest *modułowość*, o którą walczymy.

### Architektura dwuwarstwowa

Ten diagram ilustruje najważniejszy wzorzec projektowy w budowaniu systemów Agent-to-Agent. Nie mieszamy logiki biznesowej z logiką transportową.

1.  Lewa strona: *Internal Schema*
    - To jest Twoja*domena biznesowa
    - Tutaj definiujesz obiekty takie jak `TriageResult`, `TicketCategory` czy `ProductDetails`.
    - Tego używa **Instructor** i **LLM**, aby zapewnić poprawność merytoryczną ("Czy to na pewno jest kategoria 'Technical'?").
    - Zmieniając ten schemat, zmieniasz inteligencję agenta.

2. Prawa strona: *External Schema* (protokół sieciowy 📡)
    - To jest Twój **kontrakt publiczny**.
    - Tutaj definiujesz "koperty", w których przesyłane są dane: `TaskRequest`, `TaskResponse`, `AgentCapability`.
    - Tego używa **FastAPI** i **Klient HTTP**, aby zapewnić, że paczka dotrze z punktu A do B.
    - Ten schemat powinien być **stabilny** (rzadko zmieniany), aby nie psuć integracji z innymi systemami.

3. Połączenie: Enkapsulacja (zagnieżdżenie)
    - Zwróć uwagę na strzałkę "NESTED IN".
    - Wynik myślenia agenta (`result`) jest tylko jednym z pól w odpowiedzi sieciowej (`TaskResponse`).
    - Dzięki temu, jeśli Twój agent jutro zacznie zwracać dodatkowe dane (np. `sentiment_score`), nie musisz zmieniać całego protokołu HTTP – po prostu rozszerzasz wewnętrzny obiekt `result`.        
        
```
╔═══════════════════════════════════════════════════════════════════════════════════════╗
║                                                                                       ║
║   INTERNAL vs EXTERNAL SCHEMAS                                                        ║
║                                                                                       ║
║   ┌───────────────────────────────────┐    ┌───────────────────────────────────┐      ║
║   │  🧠 INTERNAL SCHEMA               │    │  📡 EXTERNAL SCHEMA (A2A)         │      ║
║   │     (Agent's Brain)               │    │     (Network Protocol)            │      ║
║   ├───────────────────────────────────┤    ├───────────────────────────────────┤      ║
║   │                                   │    │                                   │      ║
║   │  • TriageResult                   │    │  • AgentCapability (/info)        │      ║
║   │  • TicketCategory                 │    │  • TaskRequest                    │      ║
║   │  • TicketPriority                 │    │  • TaskResponse                   │      ║
║   │                                   │    │                                   │      ║
║   │  Purpose:                         │    │  Purpose:                         │      ║
║   │  Define WHAT the LLM outputs      │    │  Define HOW agents communicate    │      ║
║   │                                   │    │                                   │      ║
║   │  Used by:                         │    │  Used by:                         │      ║
║   │  Instructor + LLM                 │    │  FastAPI + HTTP                   │      ║
║   │                                   │    │                                   │      ║
║   └───────────────────────────────────┘    └───────────────────────────────────┘      ║
║                                                                                       ║
║                              ┌─────────────────┐                                      ║
║                              │   NESTED IN     │                                      ║
║                              │   TaskResponse  │                                      ║
║                              │                 │                                      ║
║                              │  result: ───────┼──► TriageResult                      ║
║                              │                 │                                      ║
║                              └─────────────────┘                                      ║
║                                                                                       ║
║   💡 This separation allows:                                                          ║
║      • Internal logic to evolve independently                                         ║
║      • A2A protocol to stay stable                                                    ║
║      • Clean contract between agents                                                  ║
║                                                                                       ║
╚═══════════════════════════════════════════════════════════════════════════════════════╝
```      

Budując system, zawsze zadaj sobie pytanie: "Czy to zmiana w logice (Internal), czy w transporcie (External)?". Jeśli zmieszasz te dwie rzeczy, każda poprawka promptu będzie wymagała aktualizacji API u wszystkich klientów. Tego chcemy uniknąć.


## Podsumowanie: wizja vs. zdrowy rozsądek

Kończymy ten moduł z głową pełną futurystycznych pojęć: `A2A`, `ACP`, `AG-UI`. To wizja, która ekscytuje i wyznacza kierunek, w którym zmierza cała branża Software 3.0. Musimy te trendy obserwować i być na bieżąco, bo to one zdefiniują architekturę systemów na najbliższą dekadę.

Jednak najważniejszą lekcją, którą warto stąd wynieść, jest **zdrowy rozsądek**.

Budowanie stabilnych systemów wymaga balansu. Nie chodzi o zgniłe kompromisy, ale o umiejętność rozróżnienia tego, co będzie możliwe "za chwilę", od tego, co działa niezawodnie "tu i teraz".
- Fakt: Standardy będą się stabilizować
- Rzeczywistość: Dzisiaj wciąż potrzebujesz własnych "pasów bezpieczeństwa"

Możesz się zablokować, czekając na idealny protokół od Google czy IBM. Ale możesz też – i do tego Cię zachęcam – dowozić wartość już teraz, używając dostępnych klocków. Traktuj poznane dzisiaj koncepty (jak podział na warstwy czy oddzielenie schematów wewnętrznych od zewnętrznych) jako **modele mentalne**. Stosuj je w projektowaniu, nawet jeśli pod spodem wciąż piszesz "zwykłe" REST API. Dzięki temu, gdy technologia dojrzeje, Twój system będzie na nią gotowy, a nie przestarzały.

### Co dalej?
Z tą wiedzą przechodzimy do **Modułu 4**, gdzie wchodzimy na **Poziom 5**. Zaczniemy rozwijać te koncepcje w bardziej złożonych scenariuszach, budując systemy, które faktycznie robią bardziej złożoną robotę. Do zobaczenia w kolejnej lekcji!

## Przydatne linki
1. [Announcing the Agent2Agent Protocol (A2A)](https://developers.googleblog.com/en/a2a-a-new-era-of-agent-interoperability/)
2. [Agent2Agent (A2A) Protocol](https://a2a-protocol.org/latest/)
3. [BeeAI now has multiple agents, and a standardized way for them to talk](https://research.ibm.com/blog/multiagent-bee-ai)
4. Docs [Agent Communication Protocol](https://agentcommunicationprotocol.dev/introduction/welcome)
5. [IBM’s Agent Communication Protocol (ACP): A technical overview for software engineers](https://workos.com/blog/ibm-agent-communication-protocol-acp)
6. [AG-UI Overview](https://docs.ag-ui.com/introduction) + [github](https://github.com/ag-ui-protocol/ag-ui)
7. [Complete Guide: Agent2Agent (A2A) Protocol - The New Standard for AI Agent Collaboration](https://dev.to/czmilo/2025-complete-guide-agent2agent-a2a-protocol-the-new-standard-for-ai-agent-collaboration-1pph)